# NNLS deconvolution of cell types

The goal of this notebook is to test the use of NNLS (non negative least squares) to deconvolve the cell type composition of a bulk mixture.
The input is a matrix containing one line per cell-type specific dmr regions and one column per target cell type:
at (i,j) the value is the mean probit for cell type j averaged on all the reads that overlap a dmr region specific to cell type i.

## 1. Import modules and set constants

In [1]:
import os
import warnings

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pickle
import pandas as pd
import json
# from tqdm.notebook import tqdm
from tqdm import tqdm

import cvxpy as cp

from scipy.stats import linregress

from EDA.edautils import plot_deconvolution_results
from methyldl.deconvolution.evaluation import compute_deconvolution_metrics
from methyldl.deconvolution.uxm import (
    load_atlas,
    uxm_deconvolution,
    rearange_uxm_deconvolution_results,
)

from methyldl.deconvolution.least_squares_deconvolvers import (
    PSLSDeconvolver,
    NNLSDeconvolver,
)
from methyldl.deconvolution.linear_calibrator import LinearCalibrator

%load_ext autoreload
%autoreload 2

We are working on the methylation atlas from [Loyfer et al.](https://www.nature.com/articles/s41586-022-05580-6), with 39 cell types.

In [2]:
# mapping from cell type names to their corresponding labels in the dataset
with open("../App/labels_dict.json", "r") as f:
    labels_dict = json.load(f)
cell_type_to_label = {v: int(k) for k, v in labels_dict.items()}
label_to_cell_type = {v: k for k, v in cell_type_to_label.items()}
N_CELL_TYPES = len(cell_type_to_label)

# constants for accessing the mixtures data
PRED_COLUMNS = [f"prediction_{i}_wavg" for i in range(N_CELL_TYPES)]
SPLIT_TO_IDX = {"train": 0, "val": 1, "test": 2}

## 2. Plotting and miscellaneous utilities

In [3]:
def get_subset(
    gt_mixtures: np.ndarray,
    subset_size: int,
    max_cell_type_in_mixtures: int = None,
    samples=None,
):
    """
    Return a random subset of mixtures constrained by composition complexity.

    Filters samples to keep only mixtures with at most `max_cell_type_in_mixtures`
    non-zero proportions in `gt_mixtures`, then randomly selects up to
    `subset_size` of those eligible samples without replacement.

    Args:
        gt_mixtures (np.ndarray): Ground-truth mixture proportions of shape
            (n_samples, n_cell_types).
        samples (np.ndarray): Sample-aligned array (predictions, features, or any
            per-sample data) with first dimension `n_samples`.
        subset_size (int): Number of eligible samples to draw.
        max_cell_type_in_mixtures (int): Maximum allowed number of present cell
            types per mixture (counted as `gt_mixtures > 0`).

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]:
            - `subset_indexes`: Indexes of selected samples in the original arrays.
            - `subset_gt_mixtures`: Selected rows from `gt_mixtures`.
            - `subset_samples`: Selected rows from `samples`.

    Raises:
        AssertionError: If `gt_mixtures` and `samples` have different lengths.
        ValueError: If no mixtures satisfy the cell-type count constraint.

    Warns:
        UserWarning: If `subset_size` exceeds the number of eligible samples; in
        that case, all eligible samples are returned.
    """
    assert samples is None or len(gt_mixtures) == len(
        samples
    ), "Mixtures and samples must have the same length."

    if max_cell_type_in_mixtures is None:
        max_cell_type_in_mixtures = gt_mixtures.shape[1]  # no filtering

    n_cell_types_per_sample = np.count_nonzero(gt_mixtures > 0, axis=1)
    eligible_indexes = np.asarray(
        n_cell_types_per_sample <= max_cell_type_in_mixtures
    ).nonzero()[0]
    if len(eligible_indexes) == 0:
        raise ValueError(
            f"No mixtures found with <= {max_cell_type_in_mixtures} cell types."
        )

    if subset_size > len(eligible_indexes):
        warnings.warn(
            f"Requested subset size {subset_size} is larger than the number of eligible mixtures {len(eligible_indexes)}. Using all eligible mixtures."
        )
        actual_subset_size = len(eligible_indexes)
    else:
        actual_subset_size = subset_size

    subset_indexes = np.random.choice(
        eligible_indexes, size=actual_subset_size, replace=False
    )

    subset_samples = samples[subset_indexes] if samples is not None else None
    subset_gt_mixtures = gt_mixtures[subset_indexes]

    return subset_indexes, subset_gt_mixtures, subset_samples

In [4]:
def plot_heatmap(
    matrix: np.ndarray,
    title: str = "Heatmap",
    color_bar_label: str = "Probability",
    xlabel="Predicted Class",
    ylabel="True Class",
    vmin: float | None = None,
    vmax: float | None = None,
):
    """Plot a heatmap of the given matrix with cell type names on the axes."""
    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax)
    plt.colorbar(label=color_bar_label)
    plt.xticks(
        ticks=np.arange(matrix.shape[1]),
        labels=[labels_dict[str(i)] for i in range(matrix.shape[1])],
        rotation=90,
    )
    plt.yticks(
        ticks=np.arange(matrix.shape[0]),
        labels=[labels_dict[str(i)] for i in range(matrix.shape[0])],
    )
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

In [5]:
def plot_mixtures_pred_vs_true(
    ground_truth_mixture: np.ndarray,
    predicted_mixtures: list[np.ndarray],
    predicted_mixture_labels: list[str],
    width: float = 0.2,
    title: str = "Predicted Mixtures vs Ground Truth Mixture",
):
    """
    Plot the predicted mixtures against the ground truth mixture for a single sample.
    The function creates a bar plot where the x axis represents the cell types and the y axis represents the mixture proportions.
    Each predicted mixture is plotted as a separate bar. The ground truth mixture is plotted as red dots for each cell type.

    Args:
        ground_truth_mixture: A 1D array of shape (n_cell_types,) representing the true mixture proportions.
        predicted_mixtures: A list of 1D arrays, each of shape (n_cell_types,), representing the predicted mixture proportions from different models.
        predicted_mixture_labels: A list of strings representing the labels for each predicted mixture (e.g., model names).
    """
    assert len(predicted_mixtures) == len(
        predicted_mixture_labels
    ), "Number of predicted mixtures must match number of labels"
    assert all(
        pred.shape == ground_truth_mixture.shape for pred in predicted_mixtures
    ), "All predicted mixtures must have the same shape as the ground truth mixture"
    plt.figure(figsize=(8, 6))
    n_cell_types = len(ground_truth_mixture)
    n_predicted_mixtures = len(predicted_mixtures)
    x = np.arange(n_cell_types)  # start of the the label locations
    x_center = (
        x + width * (n_predicted_mixtures - 1) / 2
    )  # middle of the group of bars for each cell type

    # Plot the ground truth mixture as red dots
    plt.scatter(
        x_center, ground_truth_mixture, color="red", zorder=-1, label="Ground Truth"
    )

    # Plot each predicted mixture as a bar
    for i, (predicted_mixture, label) in enumerate(
        zip(predicted_mixtures, predicted_mixture_labels)
    ):
        plt.bar(x + i * width, predicted_mixture, width, label=label)

    # plot the cell types names on the x axis, rotated by 90 degrees
    plt.xticks(x_center, cell_type_to_label.keys(), rotation=90)
    plt.ylabel("Mixture Proportions")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.tight_layout()
    plt.show()

## 3. Data Loading

### 3.1. Pure mixtures

The file that we load in this section contains the input and output data related to the pure mixtures.

For every cell_type i:

- file[i][0] - Ground Truth proportions.
- file[i][1][k] - Pandas dataframes with aggregated predictions (k in 0,1,2 means train, validation and test datasets respectfully)
- file[i][2][k] - UXM input matrices and deconvolution results (k in 0,1,2 means train, validation and test datasets respectfully)

In [6]:
# pure_mixtures_path = "../Data/mixtures/soft_labels_pure_ios.pkl"
pure_mixtures_path = "soft_labels_pure_ios.pkl"
# unpure_mixtures_first_1000_path = "../Data/mixtures/ios_deconvolution_data_hg38_grouped_dmrs_uxm_alligned_1000.pkl"
unpure_mixtures_first_1000_path = "../Tutorials/methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels/ios_full_matrices.npz"
# uxm_results_unpure_mixtures_first_1000_path = "../Data/mixtures/uxm_deconvolution_results_unpure_mixtures.pkl"
uxm_results_unpure_mixtures_first_1000_path = "../Tutorials/methylBertLoyferWithReject_attentionClassifier_DMRs_stratified_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels/uxm_deconvolution_results_unpure_mixtures_first_1000.pkl"
# atlas_path = "../Data/UXM_atlas/Atlas.U25.l4.hg38.full.tsv"
atlas_path = "../../UXM_deconv/supplemental/Atlas.U25.l4.hg38.full.tsv"

In [7]:
with open(pure_mixtures_path, "rb") as f:
    soft_labels_pure_ios = pickle.load(f)

In [8]:
# functions to extract data from the loaded pickle file containing the pure mixtures and their corresponding predictions and ground truth labels
def get_pure_prediction_matrix(
    cell_type_name: int | str, split_name: str = "train"
) -> np.ndarray:
    if isinstance(cell_type_name, str):
        cell_type_label = cell_type_to_label[cell_type_name]
    else:
        cell_type_label = cell_type_name
    split_idx = SPLIT_TO_IDX[split_name]
    return soft_labels_pure_ios[cell_type_label][1][split_idx][PRED_COLUMNS].to_numpy()


def get_pure_uxm_results(
    cell_type_name: int | str, split_name: str = "train"
) -> pd.DataFrame:
    if isinstance(cell_type_name, str):
        cell_type_label = cell_type_to_label[cell_type_name]
    else:
        cell_type_label = cell_type_name
    split_idx = SPLIT_TO_IDX[split_name]
    return np.array(soft_labels_pure_ios[cell_type_label][2][split_idx][3])


def get_pure_ground_truth_mixture(cell_type_name: int | str) -> np.ndarray:
    mixture = np.zeros(N_CELL_TYPES, dtype=float)
    if isinstance(cell_type_name, str):
        cell_type_label = cell_type_to_label[cell_type_name]
    else:
        cell_type_label = cell_type_name
    mixture[cell_type_label] = 1.0
    return mixture

### 3.2. Unpure mixtures

The file that we load in this section contains the input and output data related to the 1000 unpure mixtures.

The layout of the pkl is the same as for the pure mixtures, except that:

- the list contains 1000 tuples instead of 39
- the Ground Truth proportions are no longer one-hot vectors but rather vectors with 39 entries that sum to 1.
- the deconvolution *results* for UXM are not given and thus must be computed (the UXM *inputs* are given)

In [ ]:
if ".pkl" in unpure_mixtures_first_1000_path:
    with open(
        unpure_mixtures_first_1000_path,
        "rb",
    ) as f:
        unpure_mixtures_data = pickle.load(f)
else:
   pass

In [ ]:
# computing uxm deconvolution results for the unpure mixtures (if not already computed and saved in a pickle file)
file_uxm_unpure_results = (
    uxm_results_unpure_mixtures_first_1000_path
)
if os.path.exists(file_uxm_unpure_results):
    with open(file_uxm_unpure_results, "rb") as f:
        uxm_unpure_results = pickle.load(f)
    print(
        f"Loaded precomputed UxM deconvolution results for unpure mixtures from {file_uxm_unpure_results}"
    )
else:
    uxm_unpure_results = []
    uxm_atlas, uxm_ref_cells = load_atlas(
       atlas_path
    )
    for mixture_idx in tqdm(range(len(unpure_mixtures_data))):
        uxm_unpure_results.append([])
        for split_idx in range(3):
            if ".pkl" in unpure_mixtures_first_1000_path:
                sf, counts = unpure_mixtures_data[mixture_idx][2][split_idx][:2]
            uxm_raw = uxm_deconvolution(
                atlas=uxm_atlas,
                ref_cells=uxm_ref_cells,
                sf=sf,
                counts=counts,
                sample_names=["pseudo_balk_sample"],
            )[0]
            uxm_aligned = np.array(
                rearange_uxm_deconvolution_results(
                    labels_dict_reversed=cell_type_to_label,
                    uxm_proportions=uxm_raw,
                    ref_cells=uxm_ref_cells,
                )
            )
            uxm_unpure_results[-1].append(uxm_aligned)
    with open(file_uxm_unpure_results, "wb") as f:
        pickle.dump(uxm_unpure_results, f)
    print(
        f"Saved UxM deconvolution results for unpure mixtures to {file_uxm_unpure_results}"
    )

In [12]:
# functions to extract the unpure mixtures data from the loaded pickle file, which contains the predictions and ground truth labels for the unpure mixtures
def get_unpure_prediction_matrix(
    mixture_idx: int, split_name: str = "train"
) -> np.ndarray:
    split_idx = SPLIT_TO_IDX[split_name]
    return unpure_mixtures_data[mixture_idx][1][split_idx][PRED_COLUMNS].to_numpy()


def get_unpure_uxm_results(mixture_idx: int, split_name: str = "train") -> pd.DataFrame:
    split_idx = SPLIT_TO_IDX[split_name]
    return np.array(uxm_unpure_results[mixture_idx][split_idx])


def get_unpure_ground_truth_mixture(mixture_idx: int) -> np.ndarray:
    return np.array(unpure_mixtures_data[mixture_idx][0])

### 3.3 Mixtures with feature selection

In this section, we load a file with 100000 mixtures with feature selection using Dmytro's method with cutoff = 1.1. The features were selected based on whether feature.max()/feature.mean() > 1.1 where the statistics are computed on all mixtures in the union of the train and validation and test sets.

In [ ]:
filename = "features_1.1_cutoff.npz"
filepath = f"../Data/mixtures/{filename}"
if os.path.exists(filepath):
    print(f"Loading feature-selected raw data from {filepath}")
    feature_selected_raw_data = np.load(filepath, allow_pickle=True)
else:
    print(
        f"Feature-selected raw data file not found at {filepath}. Please make sure the file exists."
    )

In [ ]:
# find the indexes of the pure mixtures
ft_selected_pure_mixtures_indexes = []
for cell_type_idx in tqdm(range(N_CELL_TYPES)):
    ft_selected_pure_mixtures_indexes.append([])
    for i, gr_proportions in enumerate(feature_selected_raw_data["proportions"]):
        if gr_proportions[cell_type_idx] == 1.0:
            ft_selected_pure_mixtures_indexes[cell_type_idx].append(i)
    if not ft_selected_pure_mixtures_indexes[cell_type_idx]:
        print(
            f"Warning: no pure mixture found for cell type {cell_type_idx} ({labels_dict[str(cell_type_idx)]})"
        )

In [ ]:
def get_pure_mixture_ft_selected_by_split(
    split_name: str, cell_type_idx: int, pure_mixture_idx
) -> np.ndarray:
    assert split_name in SPLIT_TO_IDX, f"Invalid split name: {split_name}"
    assert pure_mixture_idx == "all" or (
        isinstance(pure_mixture_idx, int)
        and (pure_mixture_idx < len(ft_selected_pure_mixtures_indexes[cell_type_idx]))
    )
    if pure_mixture_idx == "all":
        return feature_selected_raw_data[f"features_{split_name}"][
            ft_selected_pure_mixtures_indexes[cell_type_idx], :
        ]
    return feature_selected_raw_data[f"features_{split_name}"][
        ft_selected_pure_mixtures_indexes[cell_type_idx][pure_mixture_idx], :
    ]

### 3.4. All dataset of full matrices

This file contains the full matrices for all the datasets (train, validation and test) without any feature selection.

In [ ]:
all_dataset_raw = np.load("../Data/mixtures/ios_full_matrices.npz", allow_pickle=True)

In [ ]:
# find the indexes of the pure mixtures
all_data_pure_mixtures_indexes = []
for cell_type_idx in tqdm(range(N_CELL_TYPES)):
    all_data_pure_mixtures_indexes.append([])
    for i, gr_proportions in enumerate(all_dataset_raw["proportions"]):
        if gr_proportions[cell_type_idx] == 1.0:
            all_data_pure_mixtures_indexes[cell_type_idx].append(i)
    if not all_data_pure_mixtures_indexes[cell_type_idx]:
        print(
            f"Warning: no pure mixture found for cell type {cell_type_idx} ({labels_dict[str(cell_type_idx)]})"
        )

In [ ]:
def get_pure_mixture_all_data_by_split(
    split_name: str, cell_type_idx: int, pure_mixture_idx
) -> np.ndarray:
    assert split_name in SPLIT_TO_IDX, f"Invalid split name: {split_name}"
    assert pure_mixture_idx == "all" or (
        isinstance(pure_mixture_idx, int)
        and (pure_mixture_idx < len(all_data_pure_mixtures_indexes[cell_type_idx]))
    )
    if pure_mixture_idx == "all":
        return all_dataset_raw[f"features_{split_name}"][
            all_data_pure_mixtures_indexes[cell_type_idx], :, :
        ]
    return all_dataset_raw[f"features_{split_name}"][
        all_data_pure_mixtures_indexes[cell_type_idx][pure_mixture_idx], :, :
    ]

## 4. Data Preprocessing

In [ ]:
# Preparation of the train, validation and test sets for the pure mixtures
def prepare_pure_X(split_name: str = "train"):
    X = np.array(
        [
            get_pure_prediction_matrix(cell_type_idx, split_name)
            for cell_type_idx in range(N_CELL_TYPES)
        ]
    )
    return X


y_pure = np.array(
    [
        get_pure_ground_truth_mixture(cell_type_idx)
        for cell_type_idx in range(N_CELL_TYPES)
    ]
)
# keep the original shape of the data
X_pure_train = prepare_pure_X("train")
X_pure_val = prepare_pure_X("val")
X_pure_test = prepare_pure_X("test")


# transform the targets to the cell type with the highest proportion in the ground truth mixture
y_pure_labels = np.argmax(y_pure, axis=1)

# flatten the matrices
X_pure_train_full = X_pure_train.reshape(X_pure_train.shape[0], -1)
X_pure_val_full = X_pure_val.reshape(X_pure_val.shape[0], -1)
X_pure_test_full = X_pure_test.reshape(X_pure_test.shape[0], -1)

X_pure_full_by_split = {
    "train": X_pure_train_full,
    "val": X_pure_val_full,
    "test": X_pure_test_full,
}

In [ ]:
# Preparation of the train, validation and test sets for the unpure mixtures
def prepare_unpure_X(split_name: str = "train"):
    X = np.array(
        [
            get_unpure_prediction_matrix(mixture_idx, split_name)
            for mixture_idx in range(len(unpure_mixtures_data))
        ]
    )
    return X


y_unpure = np.array(
    [
        get_unpure_ground_truth_mixture(mixture_idx)
        for mixture_idx in range(len(unpure_mixtures_data))
    ]
)

# keep the original shape of the data
X_unpure_train = prepare_unpure_X("train")
X_unpure_val = prepare_unpure_X("val")
X_unpure_test = prepare_unpure_X("test")


# transform the targets to the cell type with the highest proportion in the ground truth mixture
y_unpure_labels = np.argmax(y_unpure, axis=1)

# flattens the matrices
X_unpure_train_full = X_unpure_train.reshape(X_unpure_train.shape[0], -1)
X_unpure_val_full = X_unpure_val.reshape(X_unpure_val.shape[0], -1)
X_unpure_test_full = X_unpure_test.reshape(X_unpure_test.shape[0], -1)

X_unpure_full_by_split = {
    "train": X_unpure_train_full,
    "val": X_unpure_val_full,
    "test": X_unpure_test_full,
}

In [ ]:
# Preparation of the train, validation and test sets for the mixtures with feature selection (Dmytro's method with cutoff = 1.1)
y_pure_ft_selected = np.array(
    [
        get_pure_ground_truth_mixture(cell_type_idx)
        for cell_type_idx in range(N_CELL_TYPES)
    ]
)
y_pure_labels_ft_selected = np.argmax(y_pure_ft_selected, axis=1)

X_pure_train_avg_ft_selected = np.vstack(
    [
        get_pure_mixture_ft_selected_by_split("train", cell_type_idx, "all")
        .mean(axis=0)
        .reshape(1, -1)
        for cell_type_idx in range(N_CELL_TYPES)
    ]
)

y_all_ft_selected = feature_selected_raw_data["proportions"]

X_all_ft_selected_by_split = {
    "train": feature_selected_raw_data["features_train"],
    "val": feature_selected_raw_data["features_valid"],
    "test": feature_selected_raw_data["features_test"],
}

In [ ]:
# Preparation of the train, validation and test sets for the dataset with full matrices (compute the avg of pure matrices for each cell type for the train split)
y_pure_all_data = np.array(
    [
        get_pure_ground_truth_mixture(cell_type_idx)
        for cell_type_idx in range(N_CELL_TYPES)
    ]
)
y_pure_labels_all_data = np.argmax(y_pure_all_data, axis=1)

# To avoid recomputing the average of the pure matrices for each cell type for the train split
# every time we run the notebook, we save it in a file and load it if it already exists
filepath = "../Data/mixtures/ios_full_matrices_pure.npy"
if not os.path.exists(filepath):
    print(
        f"Computing and saving avg matrices of pure mixtures for full matrices to {filepath}"
    )
    X_pure_train_avg_all_data = np.vstack(
        [
            get_pure_mixture_all_data_by_split("train", cell_type_idx, "all")
            .mean(axis=0)
            .reshape(1, -1)
            for cell_type_idx in range(N_CELL_TYPES)
        ]
    )
    np.save(filepath, X_pure_train_avg_all_data)
    print("Done computing and saving avg matrices of pure mixtures for full matrices.")
else:
    print(f"Loading avg matrices of pure mixtures for full matrices from {filepath}")
    X_pure_train_avg_all_data = np.load(filepath)
    print("Done loading avg matrices of pure mixtures for full matrices.")

y_all_data = all_dataset_raw["proportions"]

X_all_data_by_split = {
    "train": all_dataset_raw["features_train"],
    "val": all_dataset_raw["features_valid"],
    "test": all_dataset_raw["features_test"],
}

## 5. Modeling

### 5.1 NNLS with 39 reference prediction matrices

The first idea we test is very simple: assuming that the unpure prediction matrices are linear combinations of the pure prediction matrices, we can directly apply NNLS to the unpure prediction matrices using the pure prediction matrices as reference.

In other terms, given an unpure prediction matrix $M$ of shape $(m, c)$ and $c$ pure prediction matrices $P_1, P_2, ..., P_c$, where $c$ is the number of cell types, we want to solve the following optimization problem:

$$
\min_{\{w_i\}_{1\leq i \leq c}} ||\sum_{i=1}^c w_i P_i - M||_F^2 \quad \text{subject to } w_i \ge 0 \text{ for all } i
$$

where $w_i$ is the proportion of cell type i in the unpure mixture and $||.||_F$ is the Frobenius norm.

We can reformulate this problem by flattening all the matrices and concatenating the pure prediction matrices into a single matrix $P$ of shape $(m*c, c)$ and the unpure prediction matrix $M$ into a vector of shape $(m*c, 1)$:

$$
\min_{w \in \mathbb{R}^c, w \ge 0} ||P w - M||_2^2
$$

#### 5.1.3. Evaluation of the NNLS deconvolution method on full matrices

In [ ]:
nnls_deconvolver = NNLSDeconvolver()
_ = nnls_deconvolver.fit(X_pure_train_full, y_pure_labels)

test_pure_preds, test_pure_preds_unnorm, test_pure_residuals = nnls_deconvolver.predict(
    X_pure_test_full
)
plot_heatmap(
    test_pure_preds,
    title="Predicted Mixtures for Pure Samples (Normalized) - Test Set",
    color_bar_label="Predicted Proportion",
    xlabel="Predicted Cell Type",
    ylabel="True Cell Type",
    vmin=0,
    vmax=1,
)

So the performance looks really good on the test set for pure mixtures.

In [ ]:
# Comparison of the perf of NNLS and UXM on the unpure mixtures test set
test_unpure_preds, test_unpure_preds_unnorm, test_unpure_residuals = (
    nnls_deconvolver.predict(X_unpure_test_full, n_workers=1)
)
test_unpure_metrics = compute_deconvolution_metrics(
    pred=test_unpure_preds,
    target=y_unpure,
)

test_unpure_uxm_preds = np.array(
    [
        get_unpure_uxm_results(mixture_idx, "test")
        for mixture_idx in range(len(unpure_mixtures_data))
    ]
)
uxm_metrics = compute_deconvolution_metrics(
    pred=test_unpure_uxm_preds,
    target=y_unpure,
)

pd.DataFrame(
    {
        "NNLS": test_unpure_metrics,
        "UXM": uxm_metrics,
    },
    index=test_unpure_metrics.keys(),
)

The performance on the whole unpure mixture dataset is better than UXM.

#### 5.1.4. Evaluation of the NNLS deconvolution method on feature selected vectors

Now we evaluate the performance of the method on the whole dataset of 100, 000 unpure mixtures.

In [ ]:
nnls_deconv_ft_selected = NNLSDeconvolver()
_ = nnls_deconv_ft_selected.fit(X_pure_train_avg_ft_selected, y_pure_labels_ft_selected)

In [ ]:
# predictions of the mixtures on the Val set
(
    pred_mixtures_nnls_ft_selected_val,
    pred_mixtures_unnormalized_nnls_ft_selected_val,
    fr_sel_val_residuals,
) = nnls_deconv_ft_selected.predict(X_all_ft_selected_by_split["val"], n_workers=1)

In [ ]:
# prediction of the mixtures on the Test set
(
    pred_mixtures_nnls_ft_selected_test,
    pred_mixtures_unnormalized_nnls_ft_selected_test,
    fr_sel_test_residuals,
) = nnls_deconv_ft_selected.predict(X_all_ft_selected_by_split["test"], n_workers=1)

In [ ]:
# Comparison of the perf of NNLS with normalization and without normalization on the unpure mixtures val and test sets
test_unpure_metrics_ft_selected = compute_deconvolution_metrics(
    pred=pred_mixtures_nnls_ft_selected_test, target=y_all_ft_selected
)
test_unpure_metrics_fr_selected_unnorm = compute_deconvolution_metrics(
    pred=pred_mixtures_unnormalized_nnls_ft_selected_test, target=y_all_ft_selected
)
val_unpure_metrics_ft_selected = compute_deconvolution_metrics(
    pred=pred_mixtures_nnls_ft_selected_val, target=y_all_ft_selected
)
val_unpure_metrics_fr_selected_unnorm = compute_deconvolution_metrics(
    pred=pred_mixtures_unnormalized_nnls_ft_selected_val, target=y_all_ft_selected
)
pd.DataFrame(
    {
        "NNLS test": test_unpure_metrics_ft_selected,
        "NNLS unnorm test": test_unpure_metrics_fr_selected_unnorm,
        "NNLS val": val_unpure_metrics_ft_selected,
        "NNLS unnorm val": val_unpure_metrics_fr_selected_unnorm,
    },
    index=test_unpure_metrics_ft_selected.keys(),
)

In [ ]:
# plot the NNLS deconvolution results for the unpure mixtures test set with feature selection
_, _ = plot_deconvolution_results(
    y_true=y_all_ft_selected,
    y_pred=pred_mixtures_nnls_ft_selected_test,
    labels_dict={int(k): v for k, v in labels_dict.items()},
)

In [ ]:
# plot deconvolution results for the unpure mixtures test set with feature selection
# FOR THE UNNORMALIZED PREDICTIONS: this helps identify if the normalization step
# is responsible for the bad calibration of some cell types
fig, all_metrics = plot_deconvolution_results(
    y_true=y_all_ft_selected,
    y_pred=pred_mixtures_unnormalized_nnls_ft_selected_test,
    labels_dict={int(k): v for k, v in labels_dict.items()},
)

Now we analyse whether the normalization step change significantly the predictions by 
analysis the unormalized norm of the predictions. If they differ significantly
from 1, it would mean that the normalization step is really changing the predictions.

In [ ]:
# Analysis of how hard the predictions had to be normalized to get good performance,
# by looking at the distribution of the sums of the unnormalized predicted mixture proportions
# across the samples in the test set of unpure mixtures

sums_unnorm_preds_ft_selected = pred_mixtures_unnormalized_nnls_ft_selected_test.sum(
    axis=1
)
plt.figure(figsize=(8, 6))
plt.hist(sums_unnorm_preds_ft_selected, bins="rice", color="blue", alpha=0.7)
plt.xlabel("Sum of Unnormalized Predicted Mixture Proportions")
plt.ylabel("Frequency")
plt.title(
    "Distribution of the Sums of Unnormalized Predicted Mixture Proportions\nfor Unpure Mixtures - Test Set (Feature-Selected Data)"
)
plt.grid()
plt.tight_layout()
plt.show()

We see that there is a significant part of the samples who do not sum to 1 natively, so we want to see if these specific samples are the ones for which the method performs the worst, and if there is a correlation between the sum of the predictions and the performance of the method.

In [ ]:
# compute the mse per sample on the test data for the unpure mixtures with feature selection,
# and see if there is a correlation between the mse and the sum of the unnormalized predicted mixture proportions
mse_per_sample_ft_selected = np.mean(
    (pred_mixtures_nnls_ft_selected_test - y_all_ft_selected) ** 2, axis=1
)

plt.figure(figsize=(8, 6))
plt.scatter(
    sums_unnorm_preds_ft_selected,
    mse_per_sample_ft_selected,
    color="blue",
    alpha=0.7,
    s=1,
)
plt.xlabel("Sum of Unnormalized Predicted Mixture Proportions")
plt.ylabel("MSE per Sample")
plt.title(
    "Scatter plot between the Sum of Unnormalized Predicted Mixture Proportions\nand the MSE per Sample for Unpure Mixtures (Test Set)"
)
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# same graph as above but with hover information about the sample (it is a bit expensive to compute and render,
# so it is disabled by default)
if False:
    TOP_K = 3
    PROPORTION_THRESHOLD = 0.10
    LIMIT_SUBSET_SIZE_TO_SHOW = 10000
    actual_n_points_to_show = min(
        LIMIT_SUBSET_SIZE_TO_SHOW, len(pred_mixtures_nnls_ft_selected_test)
    )

    selected_indexes_to_show = np.random.choice(
        len(pred_mixtures_nnls_ft_selected_test),
        size=actual_n_points_to_show,
        replace=False,
    )

    def format_label_summary(
        proportions: np.ndarray,
        top_k: int,
        threshold: float,
    ) -> tuple[str, str]:
        sorted_indices = np.argsort(proportions)[::-1]
        top_indices = sorted_indices[:top_k]
        threshold_indices = sorted_indices[proportions[sorted_indices] >= threshold]

        top_summary = "<br>".join(
            f"{labels_dict[str(idx)]}: {proportions[idx]:.4f}" for idx in top_indices
        )

        threshold_summary = "<br>".join(
            f"{labels_dict[str(idx)]}: {proportions[idx]:.4f}"
            for idx in threshold_indices
        )
        if not threshold_summary:
            threshold_summary = "None"

        return top_summary, threshold_summary

    mse_per_sample_ft_selected = np.mean(
        (
            pred_mixtures_nnls_ft_selected_test[selected_indexes_to_show]
            - y_all_ft_selected[selected_indexes_to_show]
        )
        ** 2,
        axis=1,
    )

    hover_rows = []
    for sample_idx, (ground_truth_mixture, predicted_mixture) in enumerate(
        zip(
            y_all_ft_selected[selected_indexes_to_show],
            pred_mixtures_nnls_ft_selected_test[selected_indexes_to_show],
        )
    ):
        gt_top_k_summary, gt_threshold_summary = format_label_summary(
            ground_truth_mixture,
            top_k=TOP_K,
            threshold=PROPORTION_THRESHOLD,
        )
        pred_top_k_summary, pred_threshold_summary = format_label_summary(
            predicted_mixture,
            top_k=TOP_K,
            threshold=PROPORTION_THRESHOLD,
        )
        hover_rows.append(
            [
                sample_idx,
                gt_top_k_summary,
                pred_top_k_summary,
                gt_threshold_summary,
                pred_threshold_summary,
            ]
        )

    fig = go.Figure(
        data=[
            go.Scatter(
                x=sums_unnorm_preds_ft_selected[selected_indexes_to_show],
                y=mse_per_sample_ft_selected,
                mode="markers",
                marker=dict(color="blue", size=2, opacity=0.7),
                customdata=np.array(hover_rows, dtype=object),
                hovertemplate=(
                    "Sample index: %{customdata[0]}<br>"
                    "Sum of unnormalized predictions: %{x:.4f}<br>"
                    "MSE per sample: %{y:.6f}<br><br>"
                    f"Top {TOP_K} ground-truth labels:<br>"
                    "%{customdata[1]}<br><br>"
                    f"Top {TOP_K} predicted labels:<br>"
                    "%{customdata[2]}<br><br>"
                    f"Ground-truth labels above {PROPORTION_THRESHOLD:.2f}:<br>"
                    "%{customdata[3]}<br><br>"
                    f"Predicted labels above {PROPORTION_THRESHOLD:.2f}:<br>"
                    "%{customdata[4]}"
                    "<extra></extra>"
                ),
            )
        ],
    )

    fig.update_layout(
        title=(
            "Correlation between the Sum of Unnormalized Predicted Mixture Proportions"
            "<br>and the MSE per Sample for Unpure Mixtures - Test Set "
            "(Feature-Selected Data)"
        ),
        xaxis_title="Sum of Unnormalized Predicted Mixture Proportions",
        yaxis_title="MSE per Sample",
        template="plotly_white",
        width=800,
        height=600,
    )

    fig.show()

### 5.2 PSLS: Probability Simplex Least Squares

Now we want to test if adding the constraint that the proportions must sum to 1 improves the performance of the method.

In other terms, we want to solve the following optimization problem (PSLS: Probability Simplex Least Squares):

$$
\min_{w \in \mathbb{R}^c} ||P w - M||_2^2 \quad \text{subject to} \quad w_i \ge 0, \sum_{i=1}^c w_i = 1
$$

Because the solvers for this problem are slower than for NNLS, we will often test the method on a subset of the data.

#### 5.2.1. Evaluation of the PSLS deconvolution method on feature selected (1.1 cutoff) vectors

Compute the results of the PSLS deconvolver on a subset of the test set of unpure mixtures with feature selection.

In [ ]:
psls_deconvolver = PSLSDeconvolver(solver_type="cvxpy")
_ = psls_deconvolver.fit(X_pure_train_avg_ft_selected, y_pure_labels)

subset_indexes, _, _ = get_subset(
    gt_mixtures=y_all_ft_selected,
    subset_size=2000,
)

test_pred_ft_sel_psls_subset = psls_deconvolver.predict(
    X_all_ft_selected_by_split["test"][subset_indexes], n_workers=2
)

compute_deconvolution_metrics(
    pred=test_pred_ft_sel_psls_subset,
    target=y_all_ft_selected[subset_indexes],
)

Compute the predictions on the whole test set and evaluate the performance of the method.

In [ ]:
if False:  # "Expensive cell to run"
    test_pred_ft_sel_psls = psls_deconvolver.predict(
        X_all_ft_selected_by_split["test"], n_workers=4
    )

    compute_deconvolution_metrics(
        pred=test_pred_ft_sel_psls,
        target=y_all_ft_selected,
    )

    _ = plot_deconvolution_results(
        y_true=y_all_ft_selected,
        y_pred=test_pred_ft_sel_psls,
        labels_dict={int(k): v for k, v in labels_dict.items()},
    )

#### 5.2.3. Evaluation of the PSLS deconvolution method on full matrices

In [ ]:
psls_deconvolver = PSLSDeconvolver()
_ = psls_deconvolver.fit(X_pure_train_avg_all_data, y_pure_labels_all_data)

# select subset
SUBSET_SIZE = 2000
subset_indexes_all_data, subset_targets_all_data, _ = get_subset(
    subset_size=SUBSET_SIZE,
    max_cell_type_in_mixtures=5,
    gt_mixtures=y_all_data,
)

# compute predictions on the subset of the validation set
val_pred_all_data_psls_subset = psls_deconvolver.predict(
    X_all_data_by_split["val"][subset_indexes_all_data].reshape(SUBSET_SIZE, -1),
    n_workers=2,
)

compute_deconvolution_metrics(
    pred=val_pred_all_data_psls_subset,
    target=subset_targets_all_data,
)

In [ ]:
_ = plot_deconvolution_results(
    y_true=y_all_data[subset_indexes_all_data],
    y_pred=val_pred_all_data_psls_subset,
    labels_dict={int(k): v for k, v in labels_dict.items()},
)

#### 5.2.5. Feature selection schemes to improve the performance

In this section, we explore how we could select features to improve the performance of the LS deconvolvers.

We base our methods on the reference matrices of the pure mixtures (obtained by averaging them on the train set).

##### 5.2.4.1. Data Exploration of feature selection

In [ ]:
# to recreate Dmytro's method:
if False:
    cutoff = 1.08
    X_pure_train_avg_all_data_ratios = X_pure_train_avg_all_data.max(
        axis=0
    ) / X_pure_train_avg_all_data.mean(axis=0)
    ft_sel_mask_cutoff = (
        (X_pure_train_avg_all_data_ratios >= cutoff).astype(int).reshape(39, 39)
    )
    plot_heatmap(
        ft_sel_mask_cutoff,
        title=f"Feature Selection Mask with Cutoff {cutoff}",
        color_bar_label="Selected (1) or Not Selected (0)",
        vmin=0,
        vmax=1,
    )

The feature selection scheme is as follows: for every feature (ie cell of the matrix), we compute the difference between
the maximum and the second highest value of this feature across the different cell types.

The rationale behind this method is that if the difference is high, it means that this feature is more specific to a particular cell type and thus more informative for the deconvolution task.

In [ ]:
# Matrix of the difference between the highest and second highest average predicted proportion for each cell type
diff_ft_matrix = (
    np.max(X_pure_train_avg_all_data, axis=0)
    - np.partition(X_pure_train_avg_all_data, -2, axis=0)[-2]
)
# Matrix of the label of the cell type with the highest value for each feature
label_ft_matrix = np.argmax(X_pure_train_avg_all_data, axis=0)

plot_heatmap(
    diff_ft_matrix.reshape(N_CELL_TYPES, N_CELL_TYPES),
    title="Difference between the Highest and Second Highest Average Feature Values across Cell Types",
    color_bar_label="Difference in Average Feature Value",
    xlabel="Predicted Cell Type",
    ylabel="DMR region",
    vmin=0,
    vmax=0.001,  # most values are very close to 0, so we set a low vmax to better visualize the differences
)

In [ ]:
# now select a cutoff value on the diff_ft_matrix matrix
# such that n_features_to_select features are selected,
n_features_to_select = (
    152  # we select the same amount as Dmytro's method for comparability
)
cutoff = np.mean(
    np.sort(diff_ft_matrix.flatten())[::-1][
        n_features_to_select - 1 : n_features_to_select + 1
    ]
)
ft_sel_mask_diff_avg = (diff_ft_matrix >= cutoff).astype(int)
print(f"Number of features selected with cutoff {cutoff}: {ft_sel_mask_diff_avg.sum()}")
plot_heatmap(
    ft_sel_mask_diff_avg.reshape(N_CELL_TYPES, N_CELL_TYPES),
    title=f"Feature Selection Mask with Cutoff {cutoff}",
    color_bar_label="Selected (1) or Not Selected (0)",
    xlabel="Predicted Cell Type",
    ylabel="DMR region",
    vmin=0,
    vmax=1,
)

In [ ]:
# feature selection map provided by Dmytro
ft_sel_mask_og = np.load("../Data/mixtures/features_mask_1.1_cutoff.npz")[
    "features_mask"
]
plot_heatmap(ft_sel_mask_og, title="Feature Selection Mask (cutoff 1.1)")

In [ ]:
# difference between the new feature selection masks and the original one provided by Dmytro
difference_mask = (
    ft_sel_mask_diff_avg.reshape(N_CELL_TYPES, N_CELL_TYPES) - ft_sel_mask_og
)
plot_heatmap(
    difference_mask
    + np.random.normal(
        scale=0.02, size=difference_mask.shape
    ),  # add a very small amount of noise to make the colors more visible
    title="Difference between Feature Selection Masks (Diff Avg vs Cutoff 1.1)",
    color_bar_label="Difference in Selection\n(1: selected by Diff Avg only, -1: selected by Cutoff 1.1 only, 0: selected by both or neither)",
    xlabel="Predicted Cell Type",
    ylabel="DMR region",
    vmin=-1,
    vmax=1,
)

In [ ]:
ft_sel_mask_diff_avg = (diff_ft_matrix >= 0.00015).astype(int)

# Discrete heatmap for label_ft_matrix with one color per label and cell-type names in colorbar + axes
unique_labels = np.sort(np.unique(label_ft_matrix)).astype(int)

# Build a discrete colormap with as many colors as labels
base_cmap = plt.get_cmap("nipy_spectral", len(unique_labels))
cmap = plt.matplotlib.colors.ListedColormap(base_cmap(np.arange(len(unique_labels))))
bounds = np.arange(len(unique_labels) + 1) - 0.5
norm = plt.matplotlib.colors.BoundaryNorm(bounds, cmap.N)

# Remap label values to 0..K-1 so the discrete colorbar is clean
label_to_pos = {lab: i for i, lab in enumerate(unique_labels)}
matrix_pos = np.vectorize(label_to_pos.get)(label_ft_matrix)

# Apply mask: set masked-out positions to NaN so they appear white
matrix_pos_masked = matrix_pos.astype(float)
matrix_pos_masked[ft_sel_mask_diff_avg == 0] = np.nan


# Robust label lookup (works whether keys are int or str)
def get_cell_type_name(label):
    return label_to_cell_type.get(label, label_to_cell_type.get(str(label), str(label)))


plt.figure(figsize=(12, 10))
im = plt.imshow(
    matrix_pos_masked.reshape(N_CELL_TYPES, N_CELL_TYPES),
    cmap=cmap,
    norm=norm,
    aspect="auto",
)

plt.title("Heatmap of the highest receiver per (emitter, region) (label-colored)")
plt.xlabel("Emitter")
plt.ylabel("Region")

# Add cell type names on axes
plt.xticks(
    ticks=np.arange(N_CELL_TYPES),
    labels=[get_cell_type_name(i) for i in range(N_CELL_TYPES)],
    rotation=90,
)
plt.yticks(
    ticks=np.arange(N_CELL_TYPES),
    labels=[get_cell_type_name(i) for i in range(N_CELL_TYPES)],
)

# Colorbar with cell type names
cbar = plt.colorbar(im, ticks=np.arange(len(unique_labels)))
cbar.set_label("Assigned cell type")
cbar.ax.set_yticklabels([get_cell_type_name(lab) for lab in unique_labels])
cbar.ax.invert_yaxis()  # reverse legend order

plt.tight_layout()
plt.show()

This visualisation should be read as follows: when you see a cell (i,j) of color c, it means that :
- when you want to predict cell type c from a read in dmr region i with signature sig, you have to take into account the fact that some signatures of reads originating from cell type j will be similar to sig.
- OR the cell type j is emmiting cross-talk signal with cell type c in dmr regions specific to cell type i
- OR for each X=emitter and each Y=region we select the highest receiver C

We could want to visialize this in another way:

- for each X=emitter and each Y=receiver we select the region Z with the highest signal

In the matrix avg_features_per_cell_type:
- axis 0 = receiver
- axis 1 = region
- axis 2 = emitter
- 
so in the first case we took the max on axis 0, now we take the max on axis 1.

In [ ]:
highest_region_per_emitter_receiver = np.argmax(
    X_pure_train_avg_all_data.reshape(-1, N_CELL_TYPES, N_CELL_TYPES), axis=1
)
diff_emission_per_emitter_receiver = (
    np.max(X_pure_train_avg_all_data.reshape(-1, N_CELL_TYPES, N_CELL_TYPES), axis=1)
    - np.partition(
        X_pure_train_avg_all_data.reshape(-1, N_CELL_TYPES, N_CELL_TYPES), -2, axis=1
    )[:, -2]
)

In [ ]:
ft_sel_mask_per_emitter_receiver = (diff_emission_per_emitter_receiver >= 0.0).astype(
    int
)

# Matrix to visualize: for each (receiver, emitter), which region has highest signal
region_matrix = highest_region_per_emitter_receiver.reshape(
    N_CELL_TYPES, N_CELL_TYPES
)  # shape: (receiver, emitter)


# Robust label lookup (works whether keys are int or str)
def get_cell_type_name(label):
    return label_to_cell_type.get(label, label_to_cell_type.get(str(label), str(label)))


unique_regions = np.sort(np.unique(region_matrix)).astype(int)
region_to_pos = {lab: i for i, lab in enumerate(unique_regions)}

# Remap region ids to 0..K-1 for a clean discrete colorbar
z = np.vectorize(region_to_pos.get)(region_matrix).astype(float)

# Apply mask: hidden cells become NaN (shown as gaps/white)
z[ft_sel_mask_per_emitter_receiver == 0] = np.nan

n_receivers, n_emitters = N_CELL_TYPES, N_CELL_TYPES
emitter_labels = [get_cell_type_name(i) for i in range(n_emitters)]
receiver_labels = [get_cell_type_name(i) for i in range(n_receivers)]

# Build hover text payload
customdata = np.empty((n_receivers, n_emitters, 3), dtype=object)
for r in range(n_receivers):
    for e in range(n_emitters):
        customdata[r, e, 0] = emitter_labels[e]
        customdata[r, e, 1] = receiver_labels[r]
        customdata[r, e, 2] = get_cell_type_name(int(region_matrix[r, e]))

fig = go.Figure(
    data=go.Heatmap(
        z=z,
        x=emitter_labels,
        y=receiver_labels,
        customdata=customdata,
        colorscale="Turbo",
        zmin=0,
        zmax=len(unique_regions) - 1,
        hoverongaps=False,
        colorbar=dict(
            title="Region with highest emission",
            tickmode="array",
            tickvals=np.arange(len(unique_regions)),
            ticktext=[get_cell_type_name(lab) for lab in unique_regions],
        ),
        hovertemplate=(
            "Emitter: %{customdata[0]}<br>"
            "Receiver: %{customdata[1]}<br>"
            "Top region: %{customdata[2]}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Heatmap of the highest region per (emitter, receiver) (interactive)",
    xaxis_title="Emitter",
    yaxis_title="Receiver",
    template="plotly_white",
    width=1000,
    height=850,
)

fig.show()

##### 5.2.4.2. Evaluation of the new feature selection scheme

In [ ]:
n_features_to_select = (
    150  # we select the same amount as Dmytro's method for comparability
)
cutoff = np.mean(
    np.sort(diff_ft_matrix.flatten())[::-1][
        n_features_to_select - 1 : n_features_to_select + 1
    ]
)
ft_sel_mask_diff_avg = (diff_ft_matrix >= cutoff).astype(int)
ft_sel_mask_diff_avg = (diff_ft_matrix >= cutoff).astype(int)
ft_sel_mask_diff_avg_indices = np.asarray(ft_sel_mask_diff_avg.flatten()).nonzero()[0]
print(f"Number of features selected with cutoff {cutoff}: {ft_sel_mask_diff_avg.sum()}")

In [ ]:
psls_deconvolver = PSLSDeconvolver()
_ = psls_deconvolver.fit(
    X_pure_train_avg_all_data[:, ft_sel_mask_diff_avg_indices].reshape(
        N_CELL_TYPES, -1
    ),
    y_pure_labels_all_data,
)

SUBSET_SIZE = 2000
subset_indexes_all_data, subset_targets_all_data, _ = get_subset(
    subset_size=SUBSET_SIZE,
    max_cell_type_in_mixtures=5,
    gt_mixtures=y_all_data,
)

val_pred_all_data_psls_subset_new_ft_sel = psls_deconvolver.predict(
    X_all_data_by_split["val"][subset_indexes_all_data].reshape(SUBSET_SIZE, -1)[
        :, ft_sel_mask_diff_avg_indices
    ],
    n_workers=2,
)

compute_deconvolution_metrics(
    pred=val_pred_all_data_psls_subset_new_ft_sel,
    target=subset_targets_all_data,
)

In [ ]:
_ = plot_deconvolution_results(
    y_true=subset_targets_all_data,
    y_pred=val_pred_all_data_psls_subset_new_ft_sel,
    labels_dict={int(k): v for k, v in labels_dict.items()},
)

The new selection feature scheme is not sufficient to correct the miscalibration by itself.

#### 5.2.4. Exploration of the possible causes of the bad calibration

First we need to determine the cell types that have bad calibration (ie there predicted proportions do not match the expected proportions).

In [ ]:
psls_deconvolver = PSLSDeconvolver()
_ = psls_deconvolver.fit(X_pure_train_avg_ft_selected, y_pure_labels)

SUBSET_SIZE = 2000
subset_indexes, subset_targets, _ = get_subset(
    gt_mixtures=y_all_ft_selected,
    subset_size=SUBSET_SIZE,
    max_cell_type_in_mixtures=5,
)

test_pred_fr_sel_psls_subset = psls_deconvolver.predict(
    X_all_ft_selected_by_split["test"][subset_indexes], n_workers=2
)

print(
    compute_deconvolution_metrics(
        pred=test_pred_fr_sel_psls_subset,
        target=subset_targets,
    )
)

_, all_metrics_test_subset = plot_deconvolution_results(
    y_true=subset_targets,
    y_pred=test_pred_fr_sel_psls_subset,
    labels_dict=label_to_cell_type,
)

In [ ]:
# create a dataframe with a column for the sample index, 39 column for the predicted proportions for each cell type, and 39 column for the ground truth proportions for each cell type
df_subset_results = pd.DataFrame(
    {
        "sample_index": subset_indexes,
        **{
            f"pred_{i}": test_pred_fr_sel_psls_subset[:, i] for i in range(N_CELL_TYPES)
        },
        **{f"true_{i}": subset_targets[:, i] for i in range(N_CELL_TYPES)},
    }
)
for i in range(N_CELL_TYPES):
    df_subset_results[f"diff_true_pred_{i}"] = (
        df_subset_results[f"true_{i}"] - df_subset_results[f"pred_{i}"]
    )

In [ ]:
diff_columns = [f"diff_true_pred_{i}" for i in range(N_CELL_TYPES)]

mean_diff = df_subset_results[diff_columns].mean(axis=0)
mean_abs_diff = df_subset_results[diff_columns].abs().mean(axis=0)

correlation_results = []
df_calibration_analysis = (
    pd.DataFrame(
        {
            "cell_type_proxy": mean_diff.index,
            "mean_diff_true_pred": mean_diff.values,
            "mean_abs_diff_true_pred": mean_abs_diff.values,
        }
    )
    .sort_values(by="mean_diff_true_pred", ascending=False)
    .reset_index(drop=True)
)

df_calibration_analysis["cell_type_label"] = df_calibration_analysis[
    "cell_type_proxy"
].apply(lambda x: int(x.split("_")[-1]))
df_calibration_analysis["cell_type_name"] = df_calibration_analysis[
    "cell_type_label"
].apply(lambda x: label_to_cell_type[x])
df_calibration_analysis["r2"] = df_calibration_analysis["cell_type_name"].map(
    all_metrics_test_subset["celltype_metrics"]["per_celltype_r2"]
)
df_calibration_analysis.drop(columns=["cell_type_proxy"], inplace=True)

df_calibration_analysis.head(5)

Now that we now which cell types are the ones with bad calibration, we test a few hypothesis:

- H1: the cell types that are under predicted (colon-fibro eg) have a reference matrix that can be expressed as a linear combination of others reference matrices (and this because colon-fibro appears as a confounder in these other matrices) -> "multicollinearity"
- H2 : the cell types that are underpredicted have reference matrix with norm larger than what is usually found in the dataset, hence it is sufficient to assign a low weight to them to reach the target, and the remaining proportions are distributed across the other cell types

To test H1, we can compute the residual of the reference matrix of all cells types after projecting it on the space spanned by the other reference matrices. We can then check if the norm of this residual is correlated with the calibration of the cell type (a low residual should be correlate with a bad calibration under H1).

In [ ]:
# project one onto the others and compute the residual
results = []
for cell_type_to_project in range(N_CELL_TYPES):
    other_cell_types = [i for i in range(N_CELL_TYPES) if i != cell_type_to_project]
    A = psls_deconvolver.reference_prediction_matrix_[:, other_cell_types]
    b = psls_deconvolver.reference_prediction_matrix_[:, cell_type_to_project]
    x = cp.Variable(A.shape[1])
    objective = cp.Minimize(cp.norm(A @ x - b, 2))
    constraints = [x >= 0, cp.sum(x) == 1]
    problem = cp.Problem(objective, constraints)
    problem.solve()
    residual_norm = np.linalg.norm(A @ x.value - b)
    results.append(
        {
            "cell_type_label": cell_type_to_project,
            "cell_type_name": label_to_cell_type[cell_type_to_project],
            "residual_norm_vs_others": residual_norm,
        }
    )

df_projection_results = pd.DataFrame(results)
df_calibration_analysis = df_calibration_analysis.merge(
    df_projection_results[["cell_type_label", "residual_norm_vs_others"]],
    on="cell_type_label",
    how="left",
)
df_calibration_analysis.sort_values(
    by="residual_norm_vs_others", ascending=True, inplace=True
)
df_calibration_analysis.head(10)

In [ ]:
# compute the correlation between the mean_diff_true_pred and the residual norm
correlation_residual_diff = df_calibration_analysis["mean_diff_true_pred"].corr(
    df_calibration_analysis["residual_norm_vs_others"]
)
correlation_residual_abs_diff = df_calibration_analysis["mean_abs_diff_true_pred"].corr(
    df_calibration_analysis["residual_norm_vs_others"]
)
correlation_residual_r2 = df_calibration_analysis["r2"].corr(
    df_calibration_analysis["residual_norm_vs_others"]
)
correlation_results.extend(
    [
        ("Residual of proj on others VS diff", correlation_residual_diff),
        ("Residual of proj on others VS abs diff", correlation_residual_abs_diff),
        ("Residual of proj on others VS r2", correlation_residual_r2),
    ]
)

In [ ]:
# also compute the pairwise cosine similarity between the columns of the reference prediction matrix,
# to see if the cell types that are less well calibrated are also more similar to other cell types in the reference prediction space
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim_matrix = cosine_similarity(psls_deconvolver.reference_prediction_matrix_.T)
plot_heatmap(
    cosine_sim_matrix,
    title="Cosine Similarity Between Cell Types in Reference Prediction Space",
    color_bar_label="Cosine Similarity",
    vmin=0,
    vmax=1,
)

To test H2, we can :

- compute the norm of the reference matrix of all cell types and check if it is correlated with the calibration of the cell type (a high norm should be correlated with a bad calibration under H2).
- for each cell type: compute the difference between the norm of the reference matrix of the cell type and the mean norm of the other matrices for this cell type and check if this difference is correlated with the calibration of the cell type (a high difference should be correlated with a bad calibration under H2).

In [ ]:
# compute the correlation between the mean_diff_true_pred and the norm of the reference vector for each cell type
results = []
for cell_type_to_inspect in range(N_CELL_TYPES):
    reference_vector = psls_deconvolver.reference_prediction_matrix_[
        :, cell_type_to_inspect
    ]
    results.append(
        {
            "cell_type_label": cell_type_to_inspect,
            "cell_type_name": label_to_cell_type[cell_type_to_inspect],
            "reference_vector_norm": np.linalg.norm(reference_vector),
        }
    )
df_reference_vector_norms = pd.DataFrame(results)
df_calibration_analysis = df_calibration_analysis.merge(
    df_reference_vector_norms[["cell_type_label", "reference_vector_norm"]],
    on="cell_type_label",
    how="left",
)
correlation_norm_diff = df_calibration_analysis["reference_vector_norm"].corr(
    df_calibration_analysis["mean_diff_true_pred"]
)
correlation_norm_abs_diff = df_calibration_analysis["reference_vector_norm"].corr(
    df_calibration_analysis["mean_abs_diff_true_pred"]
)
correlation_norm_r2 = df_calibration_analysis["reference_vector_norm"].corr(
    df_calibration_analysis["r2"]
)
correlation_results.extend(
    [
        ("Norm of reference vector VS diff", correlation_norm_diff),
        ("Norm of reference vector VS abs diff", correlation_norm_abs_diff),
        ("Norm of reference vector VS r2", correlation_norm_r2),
    ]
)

In [ ]:
# compute the correlation between the difference between the norm of the reference vector and the mean norms for the vectors for the same cell types
results = []
for cell_type_to_inspect in range(N_CELL_TYPES):
    reference_vector = psls_deconvolver.reference_prediction_matrix_[
        :, cell_type_to_inspect
    ]
    norm_reference_vector = np.linalg.norm(reference_vector)

    # now we look in the test set at all the samples where the ground truth mixture
    # is a pure mixture of the cell type we are inspecting, and compute the mean norm of
    # the feature vectors for those samples
    pure_samples_indexes = np.where(y_all_ft_selected[:, cell_type_to_inspect] == 1.0)[
        0
    ]
    mean_norm_feature_vectors = np.mean(
        np.linalg.norm(X_all_ft_selected_by_split["test"][pure_samples_indexes], axis=1)
    )

    results.append(
        {
            "cell_type_label": cell_type_to_inspect,
            "cell_type_name": label_to_cell_type[cell_type_to_inspect],
            "norm_reference_vector": norm_reference_vector,
            "mean_norm_feature_vectors": mean_norm_feature_vectors,
        }
    )
df_reference_vs_mean_norms = pd.DataFrame(results)
df_reference_vs_mean_norms["norm_diff"] = (
    df_reference_vs_mean_norms["norm_reference_vector"]
    - df_reference_vs_mean_norms["mean_norm_feature_vectors"]
)
df_calibration_analysis = df_calibration_analysis.merge(
    df_reference_vs_mean_norms[["cell_type_label", "norm_diff"]],
    on="cell_type_label",
    how="left",
)
correlation_norm_diff_mean_diff = df_calibration_analysis["norm_diff"].corr(
    df_calibration_analysis["mean_diff_true_pred"]
)
correlation_norm_diff_mean_abs_diff = df_calibration_analysis["norm_diff"].corr(
    df_calibration_analysis["mean_abs_diff_true_pred"]
)
correlation_norm_diff_r2 = df_calibration_analysis["norm_diff"].corr(
    df_calibration_analysis["r2"]
)
correlation_results.extend(
    [
        (
            "Diff of norm ref vect with vect same cell type VS diff",
            correlation_norm_diff_mean_diff,
        ),
        (
            "Diff of norm ref vect with vect same cell type VS abs diff",
            correlation_norm_diff_mean_abs_diff,
        ),
        (
            "Diff of norm ref vect with vect same cell type VS r2",
            correlation_norm_diff_r2,
        ),
    ]
)

Finally, we look at all the correlations computed in this section to see which hypothesis is the most supported by the data.

In [ ]:
# show all correlations
df_correlation_results = pd.DataFrame(
    correlation_results, columns=["Correlation", "Value"]
)
# df_correlation_results.sort_values(by="Value", ascending=False, inplace=True)
print(df_correlation_results.to_markdown(index=False, tablefmt="pretty"))

The main issue seems to be multicollinearity, as the cell types that are under predicted have a reference matrix that can be well approximated by a linear combination of the other reference matrices.

#### 5.2.6. Penalty schemes to improve the calibration of the method

Here, the idea is to add a penalty term to the optimization problem that penalizes the cell types that are NOT under predicted, in order to force the method to assign more weight to the under predicted cell types and thus improve the calibration.

In other terms, we want to solve the following optimization problem:

$$
\min_{\hat{w} \in \mathbb{R}^c} ||P \hat{w} - M||_2^2 + \hat{w}^T \alpha \quad \text{subject to} \quad \hat{w}_i \ge 0, \sum_{i=1}^c \hat{w}_i = 1
$$

with with $\alpha \in \mathbb{R}^c$ being a vector of penalties for each cell type, chosen in such a way that, ideally, it is the mnimizer of:

$$
\min_{\alpha \in \mathbb{R}^c} \sum_{w, M} || \argmin_{\hat{w} \in \mathbb{R}^c, \hat{w}_i \ge 0,1^T\hat{w} = 1} \left[ ||P \hat{w} - M||_2^2 + \hat{w}^T \alpha \right] - w||_2^2
$$

In [ ]:
## Feature selection
diff_ft_matrix = (
    np.max(X_pure_train_avg_all_data, axis=0)
    - np.partition(X_pure_train_avg_all_data, -2, axis=0)[-2]
)
n_features_to_select = 150
cutoff = np.mean(
    np.sort(diff_ft_matrix.flatten())[::-1][
        n_features_to_select - 1 : n_features_to_select + 1
    ]
)
penalty_ft_select_indexes_mask = (diff_ft_matrix >= cutoff).astype(int)
penalty_ft_select_indexes = np.asarray(
    penalty_ft_select_indexes_mask.flatten()
).nonzero()[0]

In [ ]:
# train biased PSLS deconvolver so that we can later use it as reference for the penalty values
biased_psls_deconvolver = PSLSDeconvolver()
_ = biased_psls_deconvolver.fit(
    X_pure_train_avg_all_data[:, penalty_ft_select_indexes], y_pure_labels
)

SUBSET_SIZE = 2000
penalty_subset_idx, penalty_subset_targets, _ = get_subset(
    gt_mixtures=y_all_data,
    subset_size=SUBSET_SIZE,
    max_cell_type_in_mixtures=5,
)

penalty_val_pred_all_data_subset = biased_psls_deconvolver.predict(
    X_all_data_by_split["val"][penalty_subset_idx].reshape(SUBSET_SIZE, -1)[
        :, penalty_ft_select_indexes
    ],
    n_workers=2,
)

_ = plot_deconvolution_results(
    y_true=penalty_subset_targets,
    y_pred=penalty_val_pred_all_data_subset,
    labels_dict=label_to_cell_type,
)

In [ ]:
# compute several metrics that are supposed to reflect the calibration of the predictions for each cell type
me_per_cell_type = np.mean(
    (penalty_val_pred_all_data_subset - penalty_subset_targets), axis=0
)
abs_me_per_cell_type = np.abs(me_per_cell_type)
slope_per_cell_type = np.zeros(N_CELL_TYPES)
intercept_per_cell_type = np.zeros(N_CELL_TYPES)
for i in range(N_CELL_TYPES):
    slope_per_cell_type[i], intercept_per_cell_type[i] = linregress(
        penalty_val_pred_all_data_subset[:, i],
        penalty_subset_targets[:, i],
    )[:2]
pd.DataFrame(
    {
        "cell_type": label_to_cell_type.values(),
        "mean_error": me_per_cell_type,
        "abs_mean_error": abs_me_per_cell_type,
        "slope": slope_per_cell_type,
        "1/slope": 1 / slope_per_cell_type,
    }
).sort_values(by="slope", ascending=False)

The penalty scheme we choose is to set $\alpha$ as the inverse of the slope of the
calibration curve of the cell type, so that the more a cell type is under predicted, the lower
the penalty it will receive and thus the more weight the method will assign to it.

In [ ]:
# solving the optimization problem with predefined penalties for each cell type, we tune alpha on the val set
val_penalty_predictions = []
alpha = 0.5  # global hyperparameter to control the strength of the penalty,

x = cp.Variable(N_CELL_TYPES)
sample_param = cp.Parameter(len(penalty_ft_select_indexes))
objective = cp.Minimize(
    cp.norm(biased_psls_deconvolver.reference_prediction_matrix_ @ x - sample_param, 2)
    + alpha * cp.sum(cp.multiply(1 / slope_per_cell_type, x))
)
constraints = [x >= 0, cp.sum(x) == 1]
problem = cp.Problem(objective, constraints)
for sample in tqdm(X_all_data_by_split["val"][penalty_subset_idx]):
    sample_param.value = sample.flatten()[penalty_ft_select_indexes]
    problem.solve(warm_start=True)
    val_penalty_predictions.append(x.value)
val_penalty_predictions = np.array(val_penalty_predictions)

_ = plot_deconvolution_results(
    y_true=penalty_subset_targets,
    y_pred=val_penalty_predictions,
    labels_dict=label_to_cell_type,
)

Now that we found an alpha that looks good on the validation set, we test the performance of this method on the test set with different mixtures.

In [ ]:
SUBSET_SIZE = 2000
subset_indexes_others, subset_targets_others, _ = get_subset(
    gt_mixtures=y_all_data, subset_size=SUBSET_SIZE, max_cell_type_in_mixtures=5
)

test_pred_penalty = np.zeros((SUBSET_SIZE, N_CELL_TYPES))

alpha = 0.5
x = cp.Variable(N_CELL_TYPES)
sample_param = cp.Parameter(len(penalty_ft_select_indexes))
objective = cp.Minimize(
    cp.norm(biased_psls_deconvolver.reference_prediction_matrix_ @ x - sample_param, 2)
    + alpha * cp.sum(cp.multiply(1 / slope_per_cell_type, x))
)
constraints = [x >= 0, cp.sum(x) == 1]
problem = cp.Problem(objective, constraints)
for i, sample in enumerate(tqdm(X_all_data_by_split["test"][subset_indexes_others])):
    sample_param.value = sample.flatten()[penalty_ft_select_indexes]
    problem.solve(warm_start=True)
    test_pred_penalty[i] = x.value

_ = plot_deconvolution_results(
    y_true=subset_targets_others,
    y_pred=test_pred_penalty,
    labels_dict=label_to_cell_type,
)

The penalty scheme clearly overfits on the specific biases of the validation set, and does not generalize to the test set.

#### 5.2.7. Transfer function of predicted proportions

Here, we apply a very simple transfer function to the predicted proportions of the PSLS deconvolver: for each cell type, we fit a linear regression between the predicted and the true proportions of this cell type on the validation set, and then we apply the obtained linear model to the predicted proportions of the test set.

In [ ]:
## Feature selection (we use it because otherwise the runtime is too large)
diff_ft_matrix = (
    np.max(X_pure_train_avg_all_data, axis=0)
    - np.partition(X_pure_train_avg_all_data, -2, axis=0)[-2]
)
n_features_to_select = 150
cutoff = np.mean(
    np.sort(diff_ft_matrix.flatten())[::-1][
        n_features_to_select - 1 : n_features_to_select + 1
    ]
)
transfer_ft_select_mask = (diff_ft_matrix >= cutoff).astype(int)
transfer_ft_select_indexes = np.asarray(transfer_ft_select_mask.flatten()).nonzero()[0]

In [ ]:
biased_psls_deconvolver = PSLSDeconvolver()
_ = biased_psls_deconvolver.fit(
    X_pure_train_avg_all_data[:, transfer_ft_select_indexes], y_pure_labels
)

SUBSET_SIZE = 2000
subset_indexes_all_data, subset_targets_all_data, _ = get_subset(
    subset_size=SUBSET_SIZE,
    max_cell_type_in_mixtures=5,
    gt_mixtures=y_all_data,
)

Compute performances on the train set

In [ ]:
train_pred_all_data_psls_subset = biased_psls_deconvolver.predict(
    X_all_data_by_split["train"][subset_indexes_all_data].reshape(SUBSET_SIZE, -1)[
        :, transfer_ft_select_indexes
    ],
    n_workers=2,
)
_, all_metrics_train_biased = plot_deconvolution_results(
    y_true=subset_targets_all_data,
    y_pred=train_pred_all_data_psls_subset,
    labels_dict=label_to_cell_type,
)

compute_deconvolution_metrics(
    pred=train_pred_all_data_psls_subset,
    target=subset_targets_all_data,
)

Compute the data for calibration tuning on the validation set

In [ ]:
val_pred_all_data_psls_subset = biased_psls_deconvolver.predict(
    X_all_data_by_split["val"][subset_indexes_all_data].reshape(SUBSET_SIZE, -1)[
        :, transfer_ft_select_indexes
    ],
    n_workers=2,
)
_, all_metrics_val_biased = plot_deconvolution_results(
    y_true=subset_targets_all_data,
    y_pred=val_pred_all_data_psls_subset,
    labels_dict=label_to_cell_type,
)

compute_deconvolution_metrics(
    pred=val_pred_all_data_psls_subset,
    target=subset_targets_all_data,
)

In [ ]:
# Fit the linear calibrator on the validation predictions of the biased PSLS deconvolver
linear_calibrator = LinearCalibrator()
linear_calibrator.fit(
    X=val_pred_all_data_psls_subset,
    y=subset_targets_all_data,
)
clip_norm_adjusted_val_predictions, adjusted_val_predictions = (
    linear_calibrator.predict(val_pred_all_data_psls_subset)
)

_, _ = plot_deconvolution_results(
    y_true=subset_targets_all_data,
    y_pred=clip_norm_adjusted_val_predictions,
    labels_dict=label_to_cell_type,
)

Now we apply the transfer function to the predictions on the *test* set with *different* mixtures

In [ ]:
SUBSET_SIZE = 2000
subset_indexes_all_data_other, subset_targets_all_data_other, _ = get_subset(
    subset_size=SUBSET_SIZE,
    max_cell_type_in_mixtures=5,
    gt_mixtures=y_all_data,
)

test_pred_all_data_psls_subset_other = biased_psls_deconvolver.predict(
    X_all_data_by_split["test"][subset_indexes_all_data_other].reshape(SUBSET_SIZE, -1)[
        :, transfer_ft_select_indexes
    ],
    n_workers=2,
)
clip_norm_adjusted_test_predictions_other, adjusted_test_predictions_other = (
    linear_calibrator.predict(test_pred_all_data_psls_subset_other)
)

# without adjustment
fig, metrics_test_other_raw = plot_deconvolution_results(
    y_true=subset_targets_all_data_other,
    y_pred=test_pred_all_data_psls_subset_other,
    labels_dict=label_to_cell_type,
)

# with adjustment
fig, metrics_test_other_adj = plot_deconvolution_results(
    y_true=subset_targets_all_data_other,
    y_pred=clip_norm_adjusted_test_predictions_other,
    labels_dict=label_to_cell_type,
)

pd.DataFrame(
    [metrics_test_other_raw, metrics_test_other_adj], index=["Raw", "Adjusted"]
)

## 6. Global comparison

In this section we compare the performance of

- NNLS deconvolution
- NNLS deconvolution with linear transfer function
- PSLS deconvolution
- PSLS deconvolution with linear transfer function

On

- the full test set with the new feature selection scheme
- the full test set with the old feature selection scheme
- a subset of the dataset with the full matrices without feature selection

With the following metrics:

- overall R2 score
- min r2 score across cell types
- median r2 score across cell types
- overall MAE
- overall MSE
- overall max_error

In [ ]:
from sklearn.metrics import r2_score

REQUIRED_GLOBALS = [
    "N_CELL_TYPES",
    "X_pure_train_avg_ft_selected",
    "X_pure_train_avg_all_data",
    "X_all_ft_selected_by_split",
    "X_all_data_by_split",
    "y_all_ft_selected",
    "y_all_data",
    "NNLSDeconvolver",
    "PSLSDeconvolver",
    "LinearCalibrator",
    "compute_deconvolution_metrics",
]

missing_globals = [name for name in REQUIRED_GLOBALS if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run sections 1 to 4 before this comparison cell. Missing globals: "
        + ", ".join(missing_globals)
    )

NNLS_WORKERS = 1
PSLS_WORKERS = min(4, os.cpu_count() or 1)
PSLS_CHUNK_SIZE = 100
FULL_TESTSET_EVAL_SIZE = 10000
FULL_TESTSET_NEW_FT_SEED = 43
FULL_TESTSET_OLD_FT_SEED = 44
FULL_MATRIX_SUBSET_SIZE = 2000
FULL_MATRIX_MAX_COMPLEXITY = 5
SUBSET_RANDOM_SEED = 42
REFERENCE_LABELS = np.arange(N_CELL_TYPES)


def flatten_samples(samples: np.ndarray) -> np.ndarray:
    """Flatten a batch of matrices into 2D feature vectors."""
    return samples.reshape(samples.shape[0], -1)


def get_top_feature_indices(
    reference_matrix: np.ndarray, n_features: int
) -> np.ndarray:
    """Select the features with the largest max-vs-second-max margin."""
    feature_margin = (
        np.max(reference_matrix, axis=0)
        - np.partition(reference_matrix, -2, axis=0)[-2]
    )
    top_feature_indices = np.argsort(feature_margin)[::-1][:n_features]
    return np.sort(top_feature_indices)


def get_random_subset_indexes(
    n_samples: int,
    subset_size: int,
    seed: int,
) -> np.ndarray:
    """Sample a reproducible subset of indexes without replacement."""
    actual_subset_size = min(subset_size, n_samples)
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(n_samples, size=actual_subset_size, replace=False))


def get_disjoint_subset_indexes(
    gt_mixtures: np.ndarray,
    subset_size: int,
    max_cell_type_in_mixtures: int | None = None,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray]:
    """Draw two disjoint subsets from the eligible mixtures."""
    if max_cell_type_in_mixtures is None:
        eligible_indexes = np.arange(len(gt_mixtures))
    else:
        n_present_cell_types = np.count_nonzero(gt_mixtures > 0, axis=1)
        eligible_indexes = np.flatnonzero(
            n_present_cell_types <= max_cell_type_in_mixtures
        )

    if len(eligible_indexes) < 2 * subset_size:
        raise ValueError(
            f"Need at least {2 * subset_size} eligible samples, found {len(eligible_indexes)}."
        )

    rng = np.random.default_rng(seed)
    shuffled_indexes = rng.permutation(eligible_indexes)
    return (
        shuffled_indexes[:subset_size],
        shuffled_indexes[subset_size : 2 * subset_size],
    )


def compute_summary_metrics(
    pred: np.ndarray,
    target: np.ndarray,
    per_celltype_eps: float = 1e-12,
) -> dict[str, float]:
    """Compute the comparison metrics reported in this section."""
    base_metrics = compute_deconvolution_metrics(pred=pred, target=target)
    per_celltype_r2 = []
    for cell_type_idx in range(target.shape[1]):
        true_values = target[:, cell_type_idx]
        pred_values = pred[:, cell_type_idx]
        if np.var(true_values) <= per_celltype_eps:
            per_celltype_r2.append(np.nan)
        else:
            per_celltype_r2.append(r2_score(true_values, pred_values))

    per_celltype_r2 = np.asarray(per_celltype_r2, dtype=float)
    valid_r2 = per_celltype_r2[~np.isnan(per_celltype_r2)]
    if len(valid_r2) == 0:
        raise ValueError("No valid per-cell-type R2 values could be computed.")

    return {
        "overall_r2": float(r2_score(target.reshape(-1), pred.reshape(-1))),
        "min_r2_cell_type": float(valid_r2.min()),
        "median_r2_cell_type": float(np.median(valid_r2)),
        "overall_mae": base_metrics["mae"],
        "overall_mse": base_metrics["mse"],
        "overall_max_error": base_metrics["max_error"],
    }


def fit_predict_nnls(
    reference_X: np.ndarray,
    val_X: np.ndarray,
    test_X: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Fit NNLS on the reference set and predict on val and test."""
    model = NNLSDeconvolver()
    _ = model.fit(reference_X, REFERENCE_LABELS)
    val_pred, _, _ = model.predict(val_X, n_workers=NNLS_WORKERS)
    test_pred, _, _ = model.predict(test_X, n_workers=NNLS_WORKERS)
    return val_pred, test_pred


def fit_predict_psls(
    reference_X: np.ndarray,
    val_X: np.ndarray,
    test_X: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Fit PSLS on the reference set and predict on val and test."""
    model = PSLSDeconvolver(solver_type="cvxpy")
    _ = model.fit(reference_X, REFERENCE_LABELS)
    val_pred = model.predict(
        val_X,
        n_workers=PSLS_WORKERS,
        chunk_size=PSLS_CHUNK_SIZE,
    )
    test_pred = model.predict(
        test_X,
        n_workers=PSLS_WORKERS,
        chunk_size=PSLS_CHUNK_SIZE,
    )
    return val_pred, test_pred


def apply_linear_transfer(
    val_pred: np.ndarray,
    val_target: np.ndarray,
    test_pred: np.ndarray,
) -> np.ndarray:
    """Fit the linear calibrator on val and apply it to test."""
    calibrator = LinearCalibrator()
    calibrator.fit(val_pred, val_target)
    calibrated_test_pred, _ = calibrator.predict(test_pred)
    return calibrated_test_pred


n_old_features = X_pure_train_avg_ft_selected.shape[1]
new_feature_indices = get_top_feature_indices(
    X_pure_train_avg_all_data,
    n_features=n_old_features,
)

new_ft_test_subset_idx = get_random_subset_indexes(
    n_samples=len(y_all_data),
    subset_size=FULL_TESTSET_EVAL_SIZE,
    seed=FULL_TESTSET_NEW_FT_SEED,
)
old_ft_test_subset_idx = get_random_subset_indexes(
    n_samples=len(y_all_ft_selected),
    subset_size=FULL_TESTSET_EVAL_SIZE,
    seed=FULL_TESTSET_OLD_FT_SEED,
)

full_matrix_val_subset_idx, full_matrix_test_subset_idx = get_disjoint_subset_indexes(
    gt_mixtures=y_all_data,
    subset_size=FULL_MATRIX_SUBSET_SIZE,
    max_cell_type_in_mixtures=FULL_MATRIX_MAX_COMPLEXITY,
    seed=SUBSET_RANDOM_SEED,
)

dataset_configs = [
    {
        "dataset": "10k-sample test subset - new feature selection",
        "reference_X": X_pure_train_avg_all_data[:, new_feature_indices],
        "val_X": flatten_samples(X_all_data_by_split["val"])[:, new_feature_indices],
        "val_y": y_all_data,
        "test_X": flatten_samples(X_all_data_by_split["test"][new_ft_test_subset_idx])[
            :, new_feature_indices
        ],
        "test_y": y_all_data[new_ft_test_subset_idx],
    },
    {
        "dataset": "10k-sample test subset - old feature selection",
        "reference_X": X_pure_train_avg_ft_selected,
        "val_X": X_all_ft_selected_by_split["val"],
        "val_y": y_all_ft_selected,
        "test_X": X_all_ft_selected_by_split["test"][old_ft_test_subset_idx],
        "test_y": y_all_ft_selected[old_ft_test_subset_idx],
    },
    {
        "dataset": "Subset of full matrices - no feature selection",
        "reference_X": X_pure_train_avg_all_data,
        "val_X": flatten_samples(
            X_all_data_by_split["val"][full_matrix_val_subset_idx]
        ),
        "val_y": y_all_data[full_matrix_val_subset_idx],
        "test_X": flatten_samples(
            X_all_data_by_split["test"][full_matrix_test_subset_idx]
        ),
        "test_y": y_all_data[full_matrix_test_subset_idx],
    },
]

results = []
for dataset_config in dataset_configs:
    print(f"Running {dataset_config['dataset']}...")

    nnls_val_pred, nnls_test_pred = fit_predict_nnls(
        reference_X=dataset_config["reference_X"],
        val_X=dataset_config["val_X"],
        test_X=dataset_config["test_X"],
    )
    nnls_transfer_test_pred = apply_linear_transfer(
        val_pred=nnls_val_pred,
        val_target=dataset_config["val_y"],
        test_pred=nnls_test_pred,
    )

    psls_val_pred, psls_test_pred = fit_predict_psls(
        reference_X=dataset_config["reference_X"],
        val_X=dataset_config["val_X"],
        test_X=dataset_config["test_X"],
    )
    psls_transfer_test_pred = apply_linear_transfer(
        val_pred=psls_val_pred,
        val_target=dataset_config["val_y"],
        test_pred=psls_test_pred,
    )

    for model_name, test_pred in [
        ("NNLS", nnls_test_pred),
        ("NNLS + linear transfer", nnls_transfer_test_pred),
        ("PSLS", psls_test_pred),
        ("PSLS + linear transfer", psls_transfer_test_pred),
    ]:
        results.append(
            {
                "dataset": dataset_config["dataset"],
                "model": model_name,
                **compute_summary_metrics(
                    pred=test_pred,
                    target=dataset_config["test_y"],
                ),
            }
        )

comparison_df = pd.DataFrame(results)
comparison_df["dataset"] = pd.Categorical(
    comparison_df["dataset"],
    categories=[config["dataset"] for config in dataset_configs],
    ordered=True,
)
comparison_df["model"] = pd.Categorical(
    comparison_df["model"],
    categories=[
        "NNLS",
        "NNLS + linear transfer",
        "PSLS",
        "PSLS + linear transfer",
    ],
    ordered=True,
)

comparison_df = comparison_df.sort_values(["dataset", "model"]).reset_index(drop=True)
comparison_df.round(6)

In [ ]:
comparison_df.sort_values(["overall_mae", "model", "dataset"])